# Train `dl_stereo_matching_pytorch` on Google Colab

Notebook này đã được chỉnh để bám đúng repo hiện tại `quaghien/pytorch_dl_stereo_matching`.

Cách dùng:

1. Chỉnh cell cấu hình bên dưới nếu muốn đổi chỗ lưu data hoặc output.
2. Run All.
3. Theo dõi log train ngay trong notebook hoặc ở thư mục output.

In [ ]:
#@title 1. Cấu hình
USE_GOOGLE_DRIVE = True
REPO_URL = "https://github.com/quaghien/pytorch_dl_stereo_matching.git"
REPO_DIR = "/content/pytorch_dl_stereo_matching"
DATA_ZIP = ""  # Để trống nếu zip đã nằm sẵn trong repo tại dl_stereo_matching_pytorch/data_stereo_flow.zip.
OUTPUT_ROOT = "/content/drive/MyDrive/pytorch_dl_stereo_matching_runs"

DATA_VERSION = "kitti2012"
NET_TYPE = "win37_dep9"
PATCH_SIZE = 37
DISP_RANGE = 256
OPTIMIZER = "adam"
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 5e-4
NUM_TR_IMG = 160
NUM_VAL_IMG = 34
NUM_VAL_LOC = 5000
TRAIN_SAMPLES_PER_EPOCH = 50000
BATCH_SIZE = 128
NUM_ITER = 40000
EVAL_EVERY = 100

MODEL_DIR = f"{OUTPUT_ROOT}/model_win37_kitti2012_quality"

In [ ]:
#@title 2. Mount Google Drive nếu cần
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
print('OUTPUT_ROOT =', OUTPUT_ROOT)

In [ ]:
#@title 3. Chuẩn bị repo và dependency
import subprocess
from pathlib import Path

candidate_paths = [
    Path.cwd(),
    Path.cwd().parent,
    Path(REPO_DIR),
]

repo_path = None
for candidate in candidate_paths:
    if (candidate / 'dl_stereo_matching_pytorch').exists() and (candidate / '.git').exists():
        repo_path = candidate
        break

if repo_path is None:
    repo_path = Path(REPO_DIR)
    if not repo_path.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_path)], check=True)

REPO_DIR = str(repo_path.resolve())
%cd {REPO_DIR}
%pip install -q numpy pillow matplotlib pytest

print('Repo ready at', REPO_DIR)

In [ ]:
#@title 4. Giải nén dataset
import zipfile
from pathlib import Path

repo_data_dir = Path(REPO_DIR) / 'dl_stereo_matching_pytorch' / 'data'
repo_data_dir.mkdir(parents=True, exist_ok=True)
default_zip = Path(REPO_DIR) / 'dl_stereo_matching_pytorch' / 'data_stereo_flow.zip'
data_zip = Path(DATA_ZIP) if DATA_ZIP else default_zip
if not data_zip.exists():
    raise FileNotFoundError(f'Không thấy file zip dataset tại {data_zip}. Hãy sửa biến DATA_ZIP ở cell cấu hình.')

training_dir = repo_data_dir / 'training'
if not training_dir.exists():
    with zipfile.ZipFile(data_zip, 'r') as zf:
        zf.extractall(repo_data_dir)

print('Dataset extracted to', repo_data_dir)

In [ ]:
#@title 5. Smoke test nhanh
!python -m pytest dl_stereo_matching_pytorch/tests -q

In [ ]:
#@title 6. Train preset chất lượng cao
from pathlib import Path

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
log_path = Path(MODEL_DIR) / 'train.log'

cmd = f"""
python -m dl_stereo_matching_pytorch.train \
  --data-root {repo_data_dir} \
  --util-root {repo_data_dir} \
  --model-dir {MODEL_DIR} \
  --data-version {DATA_VERSION} \
  --net-type {NET_TYPE} \
  --patch-size {PATCH_SIZE} \
  --disp-range {DISP_RANGE} \
  --optimizer {OPTIMIZER} \
  --learning-rate {LEARNING_RATE} \
  --weight-decay {WEIGHT_DECAY} \
  --num-tr-img {NUM_TR_IMG} \
  --num-val-img {NUM_VAL_IMG} \
  --num-val-loc {NUM_VAL_LOC} \
  --train-samples-per-epoch {TRAIN_SAMPLES_PER_EPOCH} \
  --batch-size {BATCH_SIZE} \
  --num-iter {NUM_ITER} \
  --eval-every {EVAL_EVERY}
""".strip()

print(cmd)
!{cmd} 2>&1 | tee {log_path}

In [ ]:
#@title 7. Evaluate sau khi train xong
eval_cmd = f"""
python -m dl_stereo_matching_pytorch.evaluate \
  --data-root {repo_data_dir} \
  --util-root {repo_data_dir} \
  --model-dir {MODEL_DIR} \
  --data-version {DATA_VERSION} \
  --net-type {NET_TYPE} \
  --patch-size {PATCH_SIZE} \
  --disp-range {DISP_RANGE} \
  --num-tr-img {NUM_TR_IMG} \
  --num-val-img {NUM_VAL_IMG} \
  --num-val-loc {NUM_VAL_LOC} \
  --batch-size 200
""".strip()
print(eval_cmd)
!{eval_cmd}

In [ ]:
#@title 8. Xem file checkpoint và log
!ls -lh {MODEL_DIR}
!tail -n 20 {MODEL_DIR}/train.log